# NQS (Neural Quantum States) — Google Colab GPU Runner

### Quick Start:
1. **Enable GPU**: Go to **`Runtime` -> `Change runtime type`** and select **`T4 GPU`** (or **`L4` / `A100`** for larger lattices). Enable **Background execution** if available so runs survive local disconnects.
2. **Configure W&B Secret (No Google Drive needed)**: Open the **Secrets** key icon in the left sidebar, add `WANDB_API_KEY` (from https://wandb.ai/authorize), and grant notebook access. Checkpoints and live energy curves sync automatically to W&B.
3. **Run All**: Click **`Runtime` -> `Run all`**. Enable `AUTO_DISCONNECT_WHEN_DONE` in Cell 4 for unattended runs so Colab releases the GPU immediately after finishing.

In [ ]:
# @title 1. Check GPU hardware and install matched JAX / Flax / NetKet dependencies
import importlib.metadata
import shutil
import subprocess
import sys

FORCE_REINSTALL = False  # @param {type:"boolean"}

def _parse_ver(pkg: str) -> tuple[int, int] | None:
    try:
        parts = importlib.metadata.version(pkg).split(".")
        return (int(parts[0]), int(parts[1]))
    except Exception:
        return None

has_gpu = shutil.which("nvidia-smi") is not None
env_ready = (
    not FORCE_REINSTALL
    and _parse_ver("NetKet") is not None
    and _parse_ver("wandb") is not None
    and (_parse_ver("beartype") or (99, 0)) < (0, 23)
    and (_parse_ver("plum-dispatch") or (99, 0)) < (2, 8)
    and _parse_ver("jax-cuda12-plugin") is None
)

if env_ready:
    print("Matched NetKet + JAX + Flax environment already installed. Skipping pip install!")
else:
    if has_gpu:
        print("NVIDIA GPU detected! Using Colab's native CUDA JAX and installing NetKet + Flax stack...")
    else:
        print("WARNING: No GPU detected (switch via Runtime -> Change runtime type -> T4 GPU). Installing CPU stack...")

    # Remove conflicting jax-cuda12 plugin if a previous run installed netket[cuda]
    if _parse_ver("jax-cuda12-plugin") is not None:
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax-cuda12-plugin", "jax-cuda12-pjrt"])

    # Install plain `netket` (not `netket[cuda]`) so Colab's preinstalled CUDA 13 JAX plugin
    # stays intact. Pin beartype<0.23 & plum-dispatch<2.8 for NetKet 3.23 compatibility.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "netket", "flax", "optax", "beartype<0.23", "plum-dispatch<2.8",
        "wandb", "pyyaml", "matplotlib",
    ])

    # Guard against stale in-memory modules (e.g. preloaded beartype 0.23) after downgrade
    stale_pkgs = [p for p in ("beartype", "flax", "jax", "netket", "plum") if p in sys.modules]
    if stale_pkgs:
        raise RuntimeError(
            f"Updated packages ({', '.join(stale_pkgs)}) were already loaded in memory. "
            "Click 'Runtime -> Restart session' once, then run all cells again!"
        )

try:
    import jax
    import flax
    import netket as nk
except ImportError as exc:
    raise RuntimeError(
        "Packages were upgraded in an already-loaded Python session. "
        "Click 'Runtime -> Restart session' and run again!"
    ) from exc

print(f"JAX    : {jax.__version__} | Devices: {jax.devices()}")
print(f"Flax   : {flax.__version__}")
print(f"NetKet : {nk.__version__}")

In [ ]:
# @title 2. Clone or pull latest NQS repository and register package path
import importlib
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/jiahao116/NQS.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
WORK_DIR = "/content/NQS"
SRC_DIR = os.path.join(WORK_DIR, "src")

if os.path.isdir(os.path.join(WORK_DIR, ".git")):
    subprocess.check_call(["git", "-C", WORK_DIR, "pull", "origin", BRANCH])
else:
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, WORK_DIR])

os.chdir(WORK_DIR)

# Register src/ directly on sys.path so the running IPython kernel finds `nqs`
# immediately without requiring a Colab runtime restart (PEP 660 .pth quirk).
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", "."])
importlib.invalidate_caches()

import nqs
print(f"Repository ready at: {os.getcwd()}")
print(f"Imported nqs from  : {nqs.__file__}")

In [ ]:
# @title 3. Configure W&B telemetry and cloud checkpoints (Drive-free by default)
import os

USE_WANDB = True            # @param {type:"boolean"}
MOUNT_GOOGLE_DRIVE = False  # @param {type:"boolean"}

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/NQS_checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"Checkpoints will be saved to Google Drive: {CHECKPOINT_DIR}")
else:
    CHECKPOINT_DIR = "/content/NQS/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print("Drive-free mode: checkpoints sync automatically to W&B Artifacts.")

if USE_WANDB:
    try:
        from google.colab import userdata
        api_key = userdata.get("WANDB_API_KEY")
        if api_key:
            os.environ["WANDB_API_KEY"] = api_key
            print("Loaded WANDB_API_KEY from Colab Secrets.")
    except Exception:
        pass
    if not os.environ.get("WANDB_API_KEY"):
        import wandb
        wandb.login()

In [ ]:
# @title 4. Run or resume NQS experiment and render convergence plot
import os
import sys
import time
from IPython.display import Image, display

WORK_DIR = "/content/NQS"
SRC_DIR = os.path.join(WORK_DIR, "src")
if os.path.isdir(WORK_DIR):
    os.chdir(WORK_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from nqs.config import load_config
from nqs.train import run_experiment

# Available experiment configs:
# - configs/tfim_1d_rbm.yaml       (Classic starter: 1D Critical TFIM + RBM + SR)
# - configs/tfim_1d_gpt.yaml       (Autoregressive: 1D Critical TFIM + Causal Transformer)
# - configs/heisenberg_1d_gpt.yaml (Advanced: 1D Heisenberg + Complex-Phase Transformer)
CONFIG_PATH = "configs/tfim_1d_rbm.yaml"  # @param ["configs/tfim_1d_rbm.yaml", "configs/tfim_1d_gpt.yaml", "configs/heisenberg_1d_gpt.yaml"]
FRESH_START = False                 # @param {type:"boolean"}
AUTO_DISCONNECT_WHEN_DONE = False   # @param {type:"boolean"}

cfg = load_config(CONFIG_PATH)
summary = run_experiment(
    cfg=cfg,
    checkpoint_dir_override=CHECKPOINT_DIR,
    no_wandb=not USE_WANDB,
    fresh_start=FRESH_START,
)

plot_path = summary.get("plot_path")
if plot_path:
    display(Image(filename=plot_path))

if AUTO_DISCONNECT_WHEN_DONE:
    print("Metrics and checkpoints uploaded. Releasing Colab runtime in 5 seconds...")
    time.sleep(5)
    from google.colab import runtime
    runtime.unassign()